# GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift

## Parameters

In [22]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = PROJECT_ROOT / "assets"

ANTIBIOTIC_ANALYSIS_LEVEL = "antibiotic"  # antibiotic, Class, Simplified categories, High level categories, Origin
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = ASSETS_DIR / "GLASS list.xlsx"
GLASS_CLEANED_PATH = ASSETS_DIR / "glass_amr_cleaned.csv"
BACTERIA_SYNONYM_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

FIT_START_YEAR = 2000
FIT_END_YEAR = 2023
NUM_FUTURE_YEARS = 2
LOESS_FRAC = 0.6
LOESS_DEGREE = 1
TAIL_DAMPING_YEARS = None

MIN_LAST_AVAILABLE_COUNT = 1
MIN_COUNT_ABS_PCT_ERROR_MEAN = 0
MAX_COUNT_ABS_PCT_ERROR_MEAN = 100
MIN_COUNTS_BACKGROUND = 2
MIN_ABS_DELTA_SI = 0.01
MIN_SI_DELTA = None

DISPLAY_NAME = True
MAX_TEXT_LABELS = 300
WIDTH = 1400
HEIGHT = 900

## Load SI And Clean GLASS List

In [23]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)
display(glass_pairs_df.head(20))

Loaded SI rows: 4,853 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 12; antibiotics/levels: 56
Saved cleaned GLASS rows: 89 to /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [24]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df)

Loaded cached rank rows: 187 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,latest_available_year,n_pair_years,current_si_source,...,post_fit_si_abs_pct_error_mean,loess_frac,loess_degree,tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Staphylococcus aureus,Cefotaxime,2023,2028,5,2000,2023,2026,26,observed,...,187.524763,0.6,1,None,ok,None,122.0,59.0,63.0,63.0
1,Acinetobacter baumannii,Tigecycline,2023,2028,5,2000,2023,2026,20,observed,...,42.937953,0.6,1,None,ok,None,62.0,118.0,-56.0,56.0
2,Salmonella enterica,Chloramphenicol,2023,2028,5,2000,2023,2026,27,observed,...,62.566739,0.6,1,None,ok,None,64.0,111.0,-47.0,47.0
3,Klebsiella pneumoniae,Nitrofurantoin,2023,2028,5,2000,2023,2026,24,observed,...,102.721682,0.6,1,None,ok,None,129.0,83.0,46.0,46.0
4,Escherichia coli,Vancomycin,2023,2028,5,2000,2023,2026,26,observed,...,78.473182,0.6,1,None,ok,None,147.0,102.0,45.0,45.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,Klebsiella pneumoniae,Vancomycin,2023,2028,5,2000,2023,2026,25,observed,...,167.130311,0.6,1,None,ok,None,156.0,156.0,0.0,0.0
183,Salmonella enterica,Gentamicin,2023,2028,5,2000,2023,2026,27,observed,...,110.110996,0.6,1,None,ok,None,176.0,176.0,0.0,0.0
184,Staphylococcus aureus,Tigecycline,2023,2028,5,2000,2023,2026,23,observed,...,321.875830,0.6,1,None,ok,None,181.0,181.0,0.0,0.0
185,Klebsiella pneumoniae,Ceftolozane/tazobactam,2023,2028,5,2000,2023,2026,26,observed,...,26.453487,0.6,1,None,ok,None,186.0,186.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [25]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")

Displayed rows: 138; GLASS matches: 40


## Display GLASS Matches In Plot

In [26]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2026,2028,79.0,353.618501,0.531363,0.581339,0.049977,72.062576,antibiotic
1,Escherichia coli,Sulfamethoxazole/trimethoprim,2026,2028,53.0,180.607343,0.550590,0.587342,0.036752,58.308754,antibiotic
2,Klebsiella pneumoniae,Ceftazidime,2026,2028,46.0,158.159324,0.177172,0.202502,0.025330,41.773026,antibiotic
3,Escherichia coli,Meropenem,2026,2028,46.0,308.747534,0.481410,0.594402,0.112993,96.192431,antibiotic
4,Escherichia coli,Ceftazidime,2026,2028,46.0,157.601651,0.240038,0.292640,0.052602,46.718260,antibiotic
5,Klebsiella pneumoniae,Colistin,2026,2028,45.0,276.148951,0.374289,0.440396,0.066106,94.465955,antibiotic
6,Klebsiella pneumoniae,Meropenem,2026,2028,44.0,214.443719,0.312353,0.342765,0.030412,68.575231,antibiotic
7,Acinetobacter baumannii,Colistin,2026,2028,43.0,119.745388,0.181469,0.187241,0.005772,35.830240,antibiotic
8,Klebsiella pneumoniae,Ciprofloxacin,2026,2028,38.0,116.337472,0.004228,0.009352,0.005124,41.758039,antibiotic
9,Pseudomonas aeruginosa,Ceftazidime,2026,2028,37.0,152.270438,0.266979,0.286336,0.019357,59.349834,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [27]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 98


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Klebsiella pneumoniae,Ceftazidime/avibactam,2026,2028,26.0,160.309880,0.377569,0.517394,0.139825,68.747822
1,Escherichia coli,Vancomycin,2026,2028,12.0,60.329357,0.039234,0.135229,0.095996,93.688384
2,Klebsiella pneumoniae,Nitrofurantoin,2026,2028,13.0,83.484503,0.096496,0.189352,0.092856,94.564060
3,Escherichia coli,Cefuroxime,2026,2028,10.0,54.544967,0.396035,0.481642,0.085607,87.433933
4,Escherichia coli,Doxycycline,2026,2028,8.0,45.441119,0.399744,0.484945,0.085201,99.056649
...,...,...,...,...,...,...,...,...,...,...
93,Acinetobacter baumannii,Sulbactam,2026,2028,13.0,18.261708,-0.134918,-0.178256,-0.043338,33.248541
94,Staphylococcus aureus,Tigecycline,2026,2028,11.0,15.343290,-0.372216,-0.416589,-0.044373,30.434722
95,Acinetobacter baumannii,Ceftolozane/tazobactam,2026,2028,4.0,18.574628,-0.861643,-0.932181,-0.070538,99.287208
96,Streptococcus pneumoniae,Azithromycin,2026,2028,4.0,5.483270,-0.398196,-0.470834,-0.072638,20.637618


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [28]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]